# Chest X-ray classification walkthrough

This project fine-tunes ResNet18 for three-class chest X-ray classification. The current result uses a deterministic split that keeps SHA-256-identical files together, selects one checkpoint by validation macro F1 and evaluates it once on the held-out test partition.

> Exact-copy grouping prevents byte-identical leakage. It does not establish patient-level independence because patient identifiers are unavailable.

In [1]:
import json
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "evidence" / "retained-results.json").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from the project or one of its subdirectories")

spec = json.loads((ROOT / "data" / "dataset-spec.json").read_text())
results = json.loads((ROOT / "evidence" / "retained-results.json").read_text())

assert results["evidence_status"] == "retained grouped-split evaluation"
assert results["split"]["split_ready"] is True
print(f"Dataset: {spec['name']} v{spec['version']}")
print(f"DOI: {results['dataset']['doi']}")
print(f"Verified images: {results['dataset']['image_count']}")
print(f"Exact-identity groups: {results['dataset']['exact_identity_group_count']}")
print(f"Grouped split ready: {results['split']['split_ready']}")

Dataset: Chest X-Ray v1
DOI: 10.17632/p5rm59k7ph.1
Verified images: 3475
Exact-identity groups: 3470
Grouped split ready: True


## Dataset

The verified Mendeley Data V1 collection contains 3,475 images across Normal, Lung Opacity and Viral Pneumonia. Images remain excluded from the repository.

![Verified dataset composition](../assets/dataset-composition.svg)

In [2]:
total = spec["expected_total"]
print("class	expected_count	share")
for label, item in spec["classes"].items():
    print(f"{label}	{item['expected_count']}	{item['expected_count'] / total:.2%}")

class	expected_count	share
Normal	1250	35.97%
Lung Opacity	1125	32.37%
Viral Pneumonia	1100	31.65%


## Leakage-resistant split

Five exact duplicate groups were found. SHA-256 identity is the only automatic grouping rule; every exact group stays within one partition. Perceptual hashes produce direct review candidates only and never form transitive split groups.

In [3]:
split = results["split"]
labels = results["provenance"]["label_order"]
print("partition	images	exact_groups	" + "	".join(labels))
for name, values in split["partitions"].items():
    counts = "	".join(str(values["class_counts"][label]) for label in labels)
    print(f"{name}	{values['images']}	{values['exact_identity_groups']}	{counts}")
print(f"cross-partition SHA-256 violations	{split['sha256_cross_partition_violation_count']}")
print(
    "perceptual review pairs (not split groups)	"
    f"{results['dataset']['perceptual_review_candidate_pair_count']}"
)

partition	images	exact_groups	Normal	Lung Opacity	Viral Pneumonia
train	2432	2429	875	787	770
validation	521	520	187	169	165
test	522	521	188	169	165
cross-partition SHA-256 violations	0
perceptual review pairs (not split groups)	3255


## Implementation

| Area | Repository implementation |
| --- | --- |
| Data checks | Dataset contract, class counts, readable-image checks and SHA-256 identity |
| Split | Deterministic 70/15/15 exact-group allocation with leakage and class-coverage audits |
| Training | ImageNet-initialised ResNet18, dropout, class-weighted loss and light rotations |
| Selection | Highest validation macro F1 with early stopping |
| Evaluation | Accuracy, balanced accuracy, macro F1, MCC, per-class metrics and calibration measures |
| Provenance | Split, config and selected-checkpoint SHA-256 digests |

## Grouped-split result

The selected checkpoint reached validation macro F1 0.8244 at epoch 7. On 522 held-out images, test macro F1 was 0.8097 and accuracy was 0.8065.

![Grouped-split model results](../assets/model-results.svg)

In [4]:
test = results["test"]
print(f"Selected epoch	{results['model']['selected_epoch']}")
print(f"Validation macro F1	{results['model']['validation']['macro_f1']:.4f}")
print("Test metrics")
for name in ("accuracy", "balanced_accuracy", "macro_f1", "matthews_correlation_coefficient"):
    print(f"{name}	{test[name]:.4f}")
print("Per-class F1")
for label, values in test["per_class"].items():
    print(f"{label}	{values['f1']:.4f}")

Selected epoch	7
Validation macro F1	0.8244
Test metrics
accuracy	0.8065
balanced_accuracy	0.8086
macro_f1	0.8097
matthews_correlation_coefficient	0.7190
Per-class F1
Normal	0.8069
Lung Opacity	0.7735
Viral Pneumonia	0.8487


## Confusion matrix, provenance and limits

The largest error count is 38 Normal images predicted as Lung Opacity. Perceptual-hash candidates remain unadjudicated review prompts, patient independence is not established, and no external population has been evaluated. This is a research benchmark, not a medical device.

In [5]:
labels = results["provenance"]["label_order"]
print(r"actual\predicted" + "	" + "	".join(labels))
for label, row in zip(labels, results["test"]["confusion_matrix"], strict=True):
    print(label + "	" + "	".join(str(value) for value in row))
print("Provenance SHA-256")
for name in ("split_file_sha256", "config_file_sha256", "checkpoint_file_sha256"):
    print(f"{name}	{results['provenance'][name]}")

actual\predicted	Normal	Lung Opacity	Viral Pneumonia
Normal	140	38	10
Lung Opacity	17	152	0
Viral Pneumonia	2	34	129
Provenance SHA-256
split_file_sha256	ada0cb2996419ebc33dc90d38c72e8b350487292bd364670923534187b239bcf
config_file_sha256	7c94dc52a9b3802326b27b02f5f2c891f38ca94a704a7ffde4f58339140e776c
checkpoint_file_sha256	5aff448ce94537a53ba2a70bc3a8314ce0a5b4beb45242fecc2f545fb55933ca
